### Single set-up only run once per session

In [141]:
!git clone -b main https://github.com/josephwargo/Enhanced_NumPy_Transformer_LM
import sys
sys.path.append('/content/Enhanced_NumPy_Transformer_LM')

fatal: destination path 'Enhanced_NumPy_Transformer_LM' already exists and is not an empty directory.


In [142]:
# stuff I didn't write
import importlib
import numpy as np
import cupy as cp
from datasets import load_dataset
import re
import json
import gc

# stuff I wrote
import Embeddings.positional_embedding as pe
import Layer_Blocks.feed_forward as ff
import Attention.attention_block as ab
import Attention.attention_head as ah
import Layer_Blocks.transformer_block as tb
import Layer_Blocks.layer_norm as ln
import base_transformer as bt
from Helper_Functions import import_word_embeddings as iwe
from Helper_Functions import import_corpus as ic

In [143]:
# loading embeddings from huggingface into cache and then returning where it is located
# NOTE: need to alter this if pulling from local & if we want to use numpy instead of cupy
embeddings_filepath = iwe.download_word_embeddings(repo_id="stanfordnlp/glove", filename_zipped="glove.6B.zip", filename_unzipped='glove.6B.300d.txt')

In [144]:
# loading corpus dataset
corpus_dataset = load_dataset("stanfordnlp/imdb")

In [145]:
# parsing to prep for batching
# constants
start_token = '<START>'
end_token = '<END>'
num_samples = 10000


# loading corpus from huggingface and then parsing
imdb_corpus = ic.parse_corpus(corpus_dataset, start_token, end_token, num_samples)

# getting dictionary of {word: index} for each word in corpus
word2ind = ic.word_to_ind(imdb_corpus, '<PAD>')

# getting list of all words
ind2word = ic.ind_to_word(imdb_corpus, '<PAD>')


# returns a cupy array of the embeddings for our corpus
embeddings = iwe.get_embeddings_for_corpus(embeddings_filepath, word2ind, 300)

In [146]:
# batching, mapping
batch_size = 4
x_batches, Y_batches = ic.batch_corpus(imdb_corpus, word2ind, batch_size=batch_size)


### Variable size checking

#### Helper functions

In [ ]:
# AI-written helper function to identify saved variables by size
import gc, sys, types

def _name_map(roots, max_depth=6, max_nodes=200_000):
    """BFS over containers, recording a dotted path for each cp.ndarray found."""
    names, seen, queue = {}, set(), list(roots)
    while queue:
        obj, path, depth = queue.pop()
        oid = id(obj)
        if oid in seen or depth > max_depth or len(seen) > max_nodes:
            continue
        seen.add(oid)
        try:
            if type(obj) is cp.ndarray:
                names.setdefault(oid, path)
                continue
            if isinstance(obj, (types.ModuleType, type)) or callable(obj):
                continue
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if isinstance(k, str) and not k.startswith('__'):
                        queue.append((v, f"{path}.{k}" if path else k, depth + 1))
            elif isinstance(obj, (list, tuple)):
                for i, v in enumerate(obj):
                    queue.append((v, f"{path}[{i}]", depth + 1))
            elif hasattr(obj, '__dict__'):
                for k, v in vars(obj).items():
                    queue.append((v, f"{path}.{k}", depth + 1))
        except ReferenceError:
            continue
    return names
def gpu_inventory(top=20, roots=None):
    gc.collect()
    f = sys._getframe(1)
    start = [(f.f_globals, "", 0), (f.f_locals, "", 0)]
    if roots:
        start += [(v, k, 0) for k, v in roots.items()]
    names = _name_map(start)

    buffers = {}   # buffer id -> [nbytes, shape, dtype, {names}]
    for o in gc.get_objects():
        if type(o) is not cp.ndarray:
            continue
        try:
            key = id(o.data.mem)
            rec = buffers.setdefault(key, [o.nbytes, o.shape, o.dtype, set()])
            if id(o) in names:
                rec[3].add(names[id(o)])
            if o.nbytes > rec[0]:
                rec[0], rec[1], rec[2] = o.nbytes, o.shape, o.dtype
        except (ReferenceError, AttributeError):
            continue

    rows = sorted(buffers.values(), key=lambda r: r[0], reverse=True)
    named = sum(r[0] for r in rows if r[3])
    total = sum(r[0] for r in rows)
    for nb, shp, dt, ns in rows[:top]:
        label = ", ".join(sorted(ns)) if ns else "<unnamed>"
        print(f"{nb/2**20:9.2f} MiB  {str(shp):24s} {str(dt):9s} {label}")
    print(f"{total/2**20:9.2f} MiB total, {named/2**20:.2f} MiB named, "
          f"{len(rows)} buffers")

In [116]:
# AI-written helper function to remove stuff from the GPU memory
def gpu_reset(*names):
    ip = get_ipython()
    ns = ip.user_ns

    # 1. drop your own bindings
    for n in names:
        ns.pop(n, None)

    # 2. IPython's output cache: Out/_oh and _, __, ___, and _NN
    ip.displayhook.flush()          # clears Out and the _N names
    for n in ('_', '__', '___'):
        ns[n] = None

    # 3. a failed cell keeps every frame in its traceback alive
    sys.last_value = sys.last_traceback = sys.last_type = None

    # 4. let pending kernels finish before their buffers can be released
    cp.cuda.Stream.null.synchronize()
    cp.cuda.Device().synchronize()

    gc.collect()
    mp, pp = cp.get_default_memory_pool(), cp.get_default_pinned_memory_pool()
    mp.free_all_blocks()
    pp.free_all_blocks()

    print(f"used  {mp.used_bytes()/2**20:8.2f} MiB")
    print(f"pool  {mp.total_bytes()/2**20:8.2f} MiB")
    free, total = cp.cuda.runtime.memGetInfo()
    print(f"device {(total-free)/2**20:8.2f} MiB in use of {total/2**20:.0f}")

#### Determining variable size and clearining vars

In [162]:
gpu_reset("test_transformer", "optimizer", "hidden_state", "logits", "arrs")#, "embeddings", "Y_batches", "x_batches", "<unnamed>")

used    254.10 MiB
pool    423.68 MiB
device   546.88 MiB in use of 14913


In [168]:
gpu_inventory(10000)#, roots={"opt": 'optimizer'})

   368.78 MiB  (4, 340, 71083)          float32   test_transformer.output_layer.hidden_state
   138.83 MiB  (71083, 512)             float32   <unnamed>
   138.83 MiB  (71083, 512)             float32   test_transformer.model_dict.output_layer_weights
   138.83 MiB  (71083, 512)             float32   test_transformer.grad_dict.output_layer_weights
    81.35 MiB  (71083, 300)             float32   test_transformer.embeddings
    14.11 MiB  (4, 8, 340, 340)         float32   test_transformer.transformer_layers.transformer_layer_0.self_attention.head.softmax_masked_score
    14.11 MiB  (4, 8, 340, 340)         float32   test_transformer.transformer_layers.transformer_layer_1.self_attention.head.softmax_masked_score
    14.11 MiB  (4, 8, 340, 340)         float32   test_transformer.transformer_layers.transformer_layer_2.self_attention.head.softmax_masked_score
     2.66 MiB  (4, 340, 512)            float32   test_transformer.input_layer.hidden_state
     2.66 MiB  (4, 340, 512)           

In [148]:
!nvidia-smi

Sun Aug 30 19:13:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             29W /   70W |     547MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Run to refresh to latest github commit

In [160]:
# Pull the latest changes for your specific branch
%cd /content/Enhanced_NumPy_Transformer_LM
!git pull origin main
%cd /content

/content/Enhanced_NumPy_Transformer_LM
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.38 KiB | 487.00 KiB/s, done.
From https://github.com/josephwargo/Enhanced_NumPy_Transformer_LM
 * branch            main       -> FETCH_HEAD
   a915e9d..2e130e9  main       -> origin/main
Updating a915e9d..2e130e9
Fast-forward
 Attention/attention_head.py  |   2 +-
 Layer_Blocks/feed_forward.py |   4 +-
 testing.ipynb                | 330 +++++++++++++++++++++++++------------------
 3 files changed, 197 insertions(+), 139 deletions(-)
/content


In [161]:
# # # reloading packages that had any changes
importlib.reload(pe)
importlib.reload(ff)
importlib.reload(ab)
importlib.reload(ah)
importlib.reload(tb)
importlib.reload(ln)
importlib.reload(bt)
importlib.reload(iwe)
importlib.reload(ic)

<module 'Helper_Functions.import_corpus' from '/content/Enhanced_NumPy_Transformer_LM/Helper_Functions/import_corpus.py'>

### Transformer Model

Training a small model for 100 batches of real text to show that loss continuously goes down

In [164]:
# transformer model
vocab_size = len(word2ind)
test_transformer = bt.transformer(
      embeddings=embeddings
    , input_layer_shape=300, input_layer_activation='relu'
    , d_model=512, hidden_layer_activations=['relu', 'relu', 'relu']
    , hidden_layer_num_heads=8
    , output_shape=vocab_size
    , clip_val=1
    , learning_rate=.01
    , optimizer='adamw'
)

In [167]:
test_transformer.train(x_batches=x_batches, Y_batches=Y_batches, num_batches=100)

Batch: 0
Loss: 11.161839485168457

Batch: 1
Loss: 11.167409896850586

Batch: 2
Loss: 11.13958740234375

Batch: 3
Loss: 11.119735717773438

Batch: 4
Loss: 11.127016067504883

Batch: 5
Loss: 11.066934585571289

Batch: 6
Loss: 11.034013748168945

Batch: 7
Loss: 11.02165699005127

Batch: 8
Loss: 11.026554107666016

Batch: 9
Loss: 10.939865112304688

Batch: 10
Loss: 11.009312629699707

Batch: 11
Loss: 11.023387908935547

Batch: 12
Loss: 10.793757438659668

Batch: 13
Loss: 10.804043769836426

Batch: 14
Loss: 10.863357543945312

Batch: 15
Loss: 10.796849250793457

Batch: 16
Loss: 10.783685684204102

Batch: 17
Loss: 10.897904396057129

Batch: 18
Loss: 10.82326889038086

Batch: 19
Loss: 10.77530288696289

Batch: 20
Loss: 10.762395858764648

Batch: 21
Loss: 10.669633865356445

Batch: 22
Loss: 10.544419288635254

Batch: 23
Loss: 10.80401611328125

Batch: 24
Loss: 10.703259468078613

Batch: 25
Loss: 10.498987197875977

Batch: 26
Loss: 10.473210334777832

Batch: 27
Loss: 10.689133644104004

Batch: 

### Memory management

In [21]:
!nvidia-smi

Sat Jul 18 20:39:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |   14287MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
del x_batches, Y_batches, embeddings, test_transformer#, padded_corpus

gc.collect()
mempool = cp.get_default_memory_pool()
pinned_mempool = cp.get_default_pinned_memory_pool()
mempool.free_all_blocks()
pinned_mempool.free_all_blocks()


In [26]:
import torch
torch.cuda.empty_cache()

In [28]:
import sys
import gc

def clear_memory_traps():
    # 1. Clear Jupyter/IPython Output History
    try:
        ipy = get_ipython()
        if ipy:
            # Clear the massive hidden output dictionaries
            for key in ['_ih', '_oh', '_dh']:
                if key in ipy.user_ns:
                    ipy.user_ns[key].clear()
            
            # Clear the recent output shortcut variables
            for key in ['_', '__', '___']:
                ipy.user_ns.pop(key, None)
    except NameError:
        pass # Not running in an IPython environment

    # 2. Clear Exception Tracebacks
    # This releases all local variables held at the moment of a crash
    sys.last_type = None
    sys.last_value = None
    sys.last_traceback = None
    
    # If IPython specifically cached a traceback, clear it
    if 'ipy' in locals() and ipy and hasattr(ipy, 'traceback'):
        ipy.traceback = None

    # 3. Clear Metric Logging Lists
    # Replace 'loss_history' with whatever list variable you used to track metrics
    if 'loss_history' in globals():
        globals()['loss_history'].clear()

    # Force garbage collection to execute immediately
    gc.collect()

clear_memory_traps()

### Graveyard

In [10]:
# import numpy as np
import cupy as np

# Assuming your transformer class is imported as bt
import base_transformer as bt 

def generate_element_wise_data(num_batches, batch_size, seq_len, vocab_size):
    x_batches = []
    Y_batches = []
    
    for _ in range(num_batches):
        # 1. Generate random token indices
        X_indices = np.random.randint(0, vocab_size, size=(batch_size, seq_len))
        
        # 2. Target is simply each token incremented by 1 (wrapped by vocab_size)
        # Shape: (batch_size, seq_len) - This perfectly satisfies your training block
        Y_batch = (X_indices + 1) % vocab_size
        
        x_batches.append(X_indices)
        Y_batches.append(Y_batch.astype(np.int32))
        
    return x_batches, Y_batches

def run_demonstration():
    # Hyperparameters tuned for fast convergence (Proof of Life)
    vocab_size = 20      
    embedding_dim = 64
    seq_len = 4          
    batch_size = 16
    num_batches = 1000   
    
    print("=" * 50)
    print("RUNNING ZERO-MODIFICATION PROOF OF CONCEPT")
    print("=" * 50)
    
    # Mocking the pre-trained embeddings matrix
    embeddings = np.random.normal(0, 0.02, size=(vocab_size, embedding_dim)).astype(np.float32)
    
    # Generate data matching your exact expected matrix shapes
    x_batches, Y_batches = generate_element_wise_data(
        num_batches=num_batches, 
        batch_size=batch_size, 
        seq_len=seq_len, 
        vocab_size=vocab_size
    )
    
    # Initialize your model exactly as it is written
    test_transformer = bt.transformer(
        embeddings=embeddings,
        input_layer_shape=embedding_dim, 
        input_layer_activation='relu',
        d_model=64,                                  
        hidden_layer_activations=['relu', 'relu'],   
        hidden_layer_num_heads=4,
        output_shape=vocab_size,                     
        learning_rate=0.001,                         
        adam=False                                    
    )
    
    # Train the Model
    print("Starting Training Loop...")
    # Your code will handle the flattening internally without crashing
    test_transformer.train(x_batches, Y_batches, num_batches=num_batches)
    
    # Evaluate Verification
    print("\n" + "=" * 50)
    print("VERIFYING INFERENCE (SLICED) RUN")
    print("=" * 50)
    
    test_X, test_Y = generate_element_wise_data(1, 1, seq_len, vocab_size)
    
    # Run a forward pass (triggers your 'else' block, which slices x[:, -1, :])
    logits, _ = test_transformer.forward_pass(test_X[0], train=False)
    
    # Logits shape from your inference block is (1, vocab_size)
    prediction = np.argmax(logits, axis=-1)[0]
    
    # For inference, we only look at the final token because of your slice
    last_input_token = test_X[0][0][-1]
    last_target_token = test_Y[0][0][-1]
    
    print(f"Full Input Sequence: {test_X[0][0]}")
    print(f"Final Input Token:   {last_input_token}")
    print(f"Expected Next Token: {last_target_token}")
    print(f"Model Prediction:    {prediction}")
    
    if prediction == last_target_token:
        print("\nSUCCESS: The model trained and inferred perfectly!")
    else:
        print("\nKeep training: The model has not fully converged yet.")

if __name__ == "__main__":
    run_demonstration()

RUNNING ZERO-MODIFICATION PROOF OF CONCEPT
Starting Training Loop...
Batch: 0
Loss: 3.934141834517358

Batch: 1
Loss: 3.369646186201866

Batch: 2
Loss: 3.755923388881413

Batch: 3
Loss: 3.5120872725794623

Batch: 4
Loss: 3.504002128654351

Batch: 5
Loss: 3.2996753908790426

Batch: 6
Loss: 3.2315640405108548

Batch: 7
Loss: 2.9842773029208

Batch: 8
Loss: 3.1792191588889693

Batch: 9
Loss: 3.0063221924538963

Batch: 10
Loss: 3.1278507140319727

Batch: 11
Loss: 3.1591242388093432

Batch: 12
Loss: 3.040559248788239

Batch: 13
Loss: 2.9545059376990865

Batch: 14
Loss: 3.264692704297406

Batch: 15
Loss: 3.0252148033836055

Batch: 16
Loss: 3.2191153254496694

Batch: 17
Loss: 3.1494254270841124

Batch: 18
Loss: 3.055907148613577

Batch: 19
Loss: 3.090598186087066

Batch: 20
Loss: 3.0150963377854634

Batch: 21
Loss: 2.963056045048962

Batch: 22
Loss: 2.951070735603851

Batch: 23
Loss: 2.9636950233205197

Batch: 24
Loss: 3.0202661044059598

Batch: 25
Loss: 2.9716292091323044

Batch: 26
Loss: 2.

In [ ]:
test_output_list = [int(x) for x in test_output[0]]

# len()
for index in test_output_list:
    print(corpus_words[index])

distanced


In [ ]:
# ONLY USE TO SAVE/LOAD MODELS
# save_file_path = 'C:/Users/josep/Desktop/Self/Learning/NLP/Transformer/Saved_Models/Test_Model_1'
# test_transformer.save_model(save_file_path)
# test_transformer.load_model(save_file_path)

In [ ]:
# test_input = cp.array(
# [
#     [1]
#     , [2]
#     , [3]
#     , [4]
# ]
# )

test_input = cp.array([[3, 4]])

test_output = test_transformer.next_token_vocab_index(test_input)
# word2ind["hello"]